# AMGPN Per-Class t-SNE Inference

基于现有 `inference()` 配置，加载指定模型权重，执行多任务推理，并把每个任务中各类别的 support / query / prototype t-SNE 子图保存到 `results/per_class_tSNEpic`。

你可以在后面的配置单元里修改 `task_index` 和 `class_ids` 来切换任务与类别子集。

In [ ]:
from private_config import require as _cfg_require, resolve as _cfg_resolve, format_value as _cfg_format
from pathlib import Path
import sys
import json
import csv
import importlib
import pandas as pd
import torch
from torch.utils.data import DataLoader

ROOT = Path(_cfg_require('per_class_tSNEpic.ipynb.cell1.module.path'))
sys.path.append(str(ROOT))

from data_loader_clean import UAVDataset, MetaDataset
import GPN_V4_adaMulti_clean_clustering as gpn_module
importlib.reload(gpn_module)
from GPN_V4_adaMulti_clean_clustering import GPN_Optimized, GPNTrainer

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATASET_PATH = _cfg_require('per_class_tSNEpic.ipynb.cell1.module.DATASET_PATH')
MODEL_PATH = _cfg_require('per_class_tSNEpic.ipynb.cell1.module.MODEL_PATH')
OUTPUT_DIR = ROOT / 'results' / 'per_class_tSNEpic'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

N_WAY = _cfg_require('per_class_tSNEpic.ipynb.cell1.module.N_WAY')
K_SHOT = _cfg_require('per_class_tSNEpic.ipynb.cell1.module.K_SHOT')
Q_QUERY = _cfg_require('per_class_tSNEpic.ipynb.cell1.module.Q_QUERY')
NUM_TASKS = _cfg_require('per_class_tSNEpic.ipynb.cell1.module.NUM_TASKS')
N_TRIALS = _cfg_require('per_class_tSNEpic.ipynb.cell1.module.N_TRIALS')
SEED = _cfg_require('per_class_tSNEpic.ipynb.cell1.module.SEED')
TASK_INDEX = _cfg_require('per_class_tSNEpic.ipynb.cell1.module.TASK_INDEX')
TASK_INDICES_TO_PLOT = _cfg_require('per_class_tSNEpic.ipynb.cell1.module.TASK_INDICES_TO_PLOT')
CLASS_IDS = None
INCLUDE_SUPPORT = True

print('device:', DEVICE)
print('output dir:', OUTPUT_DIR)
print('model path:', MODEL_PATH)

In [ ]:
from private_config import require as _cfg_require, resolve as _cfg_resolve, format_value as _cfg_format
print('loading test dataset...')
test_dataset = UAVDataset(
    data_dir_path=DATASET_PATH,
    seed=SEED,
    is_train=False,
    train_ratio=_cfg_require('per_class_tSNEpic.ipynb.cell2.module.train_ratio'),
    max_sample_count=_cfg_require('per_class_tSNEpic.ipynb.cell2.module.max_sample_count'),
)

meta_test_dataset = MetaDataset(
    test_dataset,
    N_WAY,
    K_SHOT,
    Q_QUERY,
    num_tasks=NUM_TASKS,
    seed=SEED,
)

print('loading model...')
model = GPN_Optimized(
    # Historical configuration omitted; provide values through private configuration.
    reduction=_cfg_require('per_class_tSNEpic.ipynb.cell2.module.reduction'),
)

trainer = GPNTrainer(
    model,
    device=DEVICE,
    mlr=_cfg_require('per_class_tSNEpic.ipynb.cell2.module.mlr'),
    use_Mdistance=_cfg_require('per_class_tSNEpic.ipynb.cell2.module.use_Mdistance'),
    use_multi=_cfg_require('per_class_tSNEpic.ipynb.cell2.module.use_multi'),
    save_middle=_cfg_require('per_class_tSNEpic.ipynb.cell2.module.save_middle'),
    crop_ratio_h=_cfg_require('per_class_tSNEpic.ipynb.cell2.module.crop_ratio_h'),
    crop_ratio_l=_cfg_require('per_class_tSNEpic.ipynb.cell2.module.crop_ratio_l'),
    resize=_cfg_require('per_class_tSNEpic.ipynb.cell2.module.resize'),
    merge_threshold=_cfg_require('per_class_tSNEpic.ipynb.cell2.module.merge_threshold'),
    merge_method=_cfg_require('per_class_tSNEpic.ipynb.cell2.module.merge_method'),
)
trainer.load_model(MODEL_PATH)
trainer.model.eval()

print('dataset size:', len(test_dataset))
print('episodes:', len(meta_test_dataset))

In [ ]:
print('running full evaluation...')
evaluation_results = trainer.full_evaluation(
    test_dataset,
    n_trials=N_TRIALS,
    q_query=Q_QUERY,
)

summary_records = []
for episode_key, metrics in evaluation_results.items():
    n_way_str, k_shot_str = episode_key.split('w')
    k_shot_str = k_shot_str.replace('s', '')
    summary_records.append({
        'episode': episode_key,
        'n_way': int(n_way_str),
        'k_shot': int(k_shot_str),
        'mean_accuracy': float(metrics[0]),
        'std_accuracy': float(metrics[1]),
    })

evaluation_df = pd.DataFrame(summary_records).sort_values(['n_way', 'k_shot'], ascending=[False, False])
evaluation_csv_path = OUTPUT_DIR / 'evaluation_metrics.csv'
evaluation_json_path = OUTPUT_DIR / 'evaluation_metrics.json'
evaluation_df.to_csv(evaluation_csv_path, index=False)

with evaluation_json_path.open('w', encoding='utf-8') as f:
    json.dump(summary_records, f, ensure_ascii=False, indent=2)

print('saved evaluation csv:', evaluation_csv_path)
print('saved evaluation json:', evaluation_json_path)
display(evaluation_df)

In [ ]:
from private_config import require as _cfg_require, resolve as _cfg_resolve, format_value as _cfg_format
from pathlib import Path

print('rendering per-class t-SNE plots...')

TASK_INDICES_TO_PLOT = globals().get('TASK_INDICES_TO_PLOT', [0, 1, 2])
CLASS_IDS = globals().get('CLASS_IDS', None)
INCLUDE_SUPPORT = globals().get('INCLUDE_SUPPORT', True)
OUTPUT_DIR = globals().get('OUTPUT_DIR', Path(_cfg_require('per_class_tSNEpic.ipynb.cell4.module.path')))
N_WAY = globals().get('N_WAY', 8)
K_SHOT = globals().get('K_SHOT', 10)
Q_QUERY = globals().get('Q_QUERY', 100)
NUM_TASKS = globals().get('NUM_TASKS', 50)

if 'trainer' not in globals() or 'test_dataset' not in globals():
    raise NameError('trainer or test_dataset is not defined. Run the earlier setup cells first.')

tsne_runs = []

for task_index in TASK_INDICES_TO_PLOT:
    task_png_path = OUTPUT_DIR / f'task_{int(task_index):03d}_per_class_tsne.png'
    task_result = trainer.plot_task_tsne_grid(
        test_dataset=test_dataset,
        n_way=N_WAY,
        k_shot=K_SHOT,
        q_query=Q_QUERY,
        num_tasks=NUM_TASKS,
        task_index=int(task_index),
        class_ids=CLASS_IDS,
        include_support=INCLUDE_SUPPORT,
        save_path=str(task_png_path),
    )
    tsne_runs.append({
        'task_index': int(task_index),
        'save_path': str(task_png_path),
        'class_ids': [int(x) for x in task_result['class_ids']],
        'selected_classes': [int(x) for x in (task_result['selected_classes'].tolist() if hasattr(task_result['selected_classes'], 'tolist') else task_result['selected_classes'])],
    })

manifest_path = OUTPUT_DIR / 'tsne_manifest.json'
with manifest_path.open('w', encoding='utf-8') as f:
    json.dump(tsne_runs, f, ensure_ascii=False, indent=2)

print('saved t-SNE manifest:', manifest_path)
display(pd.DataFrame(tsne_runs))

## 结果说明

- `evaluation_metrics.csv`：`full_evaluation()` 的汇总指标
- `evaluation_metrics.json`：同一份指标的 JSON 版本
- `tsne_manifest.json`：每个任务的 t-SNE 图保存记录
- `task_XXX_per_class_tsne.png`：每个任务的 per-class 子图结果

如果你要换任务，只需要修改 `TASK_INDICES_TO_PLOT`。
如果你只想看部分类别，设置 `CLASS_IDS = [<configured>, <configured>, <configured>]` 之类即可。
如果你要换输出目录，改 `OUTPUT_DIR`。

In [ ]:
from private_config import require as _cfg_require, resolve as _cfg_resolve, format_value as _cfg_format
from evaluation_module_single_class_heatmap import plot_amgpn_assignment_heatmap
# 批量生成指定 Task 列表和 类别 列表的热力图
# 例如：生成 Task <configured>,<configured>,<configured> 中，第 <configured> 类和第 <configured> 类的热力图
plot_multiple_amgpn_assignment_heatmaps(
    trainer=trainer,
    test_dataset=test_dataset,
    task_indices=_cfg_require('per_class_tSNEpic.ipynb.cell6.module.task_indices'),       # 遍历 <configured> 个 Task
    target_class_local_ids=_cfg_require('per_class_tSNEpic.ipynb.cell6.module.target_class_local_ids'),# 每个 Task 中只画第 <configured> 类和第 <configured> 类
    n_way=N_WAY,
    k_shot=K_SHOT,
    q_query=Q_QUERY,
    num_tasks=NUM_TASKS,
    seed=SEED,
    temp=0.3,                     # 温度调低，使块状结构更明显
    save_dir=str(OUTPUT_DIR / 'multi_class_heatmaps')
)
